In [1]:
import pandas as pd
import numpy as np
import datetime
import webbrowser
import os
import plotly
import plotly.express as px
import plotly.io as pio
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error,r2_score
from nltk.sentiment.vader import SentimentIntensityAnalyzer
import nltk
from plotly.subplots import make_subplots

In [2]:
#Read the CSV files 
App_df=pd.read_csv('Play Store Data.csv')
df_review=pd.read_csv('User Reviews.csv')

In [3]:
#Data Cleaning
#Drop the column having the Null Values for 'Rating' Column
App_df=App_df.dropna(subset=['Rating'])
#Replace the null values with mode othat column
for column in App_df.columns:
    App_df[column].fillna(App_df[column].mode()[0],inplace=True)
App_df.drop_duplicates(inplace=True)   #Drop dulicates 
App_df=App_df=App_df[App_df['Rating']<=5] #Rating should be in between 1-5,so filtering the columns which are haveing the rating inbetween 1 to 5. 

C:\Users\Mayura\AppData\Local\Temp\ipykernel_21540\4126325548.py:6: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  App_df[column].fillna(App_df[column].mode()[0],inplace=True)
C:\Users\Mayura\AppData\Local\Temp\ipykernel_21540\4126325548.py:6: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chai

In [4]:
App_df=App_df[App_df['Price']>0]


In [5]:
App_df.dtypes



App                   str
Category              str
Rating            float64
Reviews               str
Size                  str
Installs              str
Type                  str
Price               int64
Content Rating        str
Genres                str
Last Updated          str
Current Ver           str
Android Ver           str
dtype: object

In [6]:
#removing the reviews which are having missing values for Translated_review column
df_review.dropna(subset=['Translated_Review'])
#Convert Review column into int
App_df['Reviews'] = App_df['Reviews'].astype(int)
# Convert the installs to int and replace the '+ 'and ',' with ' '
App_df['Installs']=App_df['Installs'].str.replace(',','').str.replace('+','').astype(int)
#Convert Price column to numeric after removing $

#App_df['Price']=App_df['Price'].str.replace('$','').astype(float)


In [7]:
#merge the data 
merge_df=pd.merge(App_df,df_review,on='App',how='inner')


In [8]:
#Data Transformation
#functions to replace the size in MB and convert into the float
def convert_size(size):
 if 'M' in size :
      return float(size.replace("M",''))
 elif 'K' in size:
      return float(size.replace('K',''))/1024
 else: 
      return np.nan#
   
App_df['Size']=App_df['Size'].apply(convert_size)


In [9]:
#logrithm
App_df['log.installs']=np.log(App_df['Installs'])
App_df['log.reviews']=np.log(App_df['Reviews'])

In [10]:
# Added a columns accoring to the rating value
def rating_group(rating):
 if rating>=4:
     return "Top Rated App"
 elif rating>=3:
      return "Above Avrage"
 else:
      return "Below Avrage"
App_df['Rating Group']=App_df['Rating'].apply(rating_group)     
     

In [11]:
#create a matrix i.e add a column to show the revenue of application
App_df['Revenue']=App_df['Price']*App_df['Installs']

In [12]:
sia=SentimentIntensityAnalyzer()

In [13]:
review = "This app is amazing! I love the new features."
sentiment_score= sia.polarity_scores(review)
print(sentiment_score)


{'neg': 0.0, 'neu': 0.42, 'pos': 0.58, 'compound': 0.8516}


In [14]:
review = "This app is very bad! I hate the new features."
sentiment_score= sia.polarity_scores(review)
print(sentiment_score)

{'neg': 0.535, 'neu': 0.465, 'pos': 0.0, 'compound': -0.8427}


In [15]:
review = "This app is okay."
sentiment_score= sia.polarity_scores(review)
print(sentiment_score)

{'neg': 0.0, 'neu': 0.612, 'pos': 0.388, 'compound': 0.2263}


In [16]:
 #Sentiment Analysis

df_review['Sentiment_Score'] = df_review['Translated_Review'].apply(lambda x: sia.polarity_scores(str(x))['compound'])

In [17]:
#Exatract the year from Last Updated column
App_df['Last Updated']=pd.to_datetime(App_df['Last Updated'],errors='coerce')
App_df['Year']=App_df['Last Updated'].dt.year

In [18]:
# Define the path for your HTML files
html_files_path = "./"

# Make sure the directory exists
if not os.path.exists(html_files_path):
    os.makedirs(html_files_path)
   

In [19]:
App_df['Last Updated']=pd.to_datetime(App_df['Last Updated'],errors='coerce')
App_df['Month']=App_df['Last Updated'].dt.month.astype(int)


In [20]:
html_files_path="./"
if not os.path.exists(html_files_path):
    os.makedirs(html_files_path)

In [21]:
# Initialize plot_containers
plot_containers = ""

In [22]:


# Initialize plot_containers
plot_containers = ""
# Save each Plotly figure to an HTML file
def save_plot_as_html(fig, filename, insight):
    global plot_containers
    filepath = os.path.join(html_files_path, filename)
    html_content = pio.to_html(fig, full_html=False, include_plotlyjs='inline')
    # Append the plot and its insight to plot_containers
    plot_containers += f"""
    <div class="plot-container" id="{filename}" onclick="openPlot('{filename}')">
        <div class="plot">{html_content}</div>
        <div class="insights">{insight}</div>
    </div>
    """
    fig.write_html(filepath, full_html=False, include_plotlyjs='inline')
   

# Define your plots
plot_width = 550
plot_height = 335
plot_bg_color = 'black'
text_color = 'white'
title_font = {'size': 16}
axis_font = {'size': 12}


In [23]:
#Task 1.Stacked Group Bar of Categaries Vs Reviews and Avarage Rating.html

import plotly.graph_objects as go
from datetime import datetime
import pytz
ist = pytz.timezone("Asia/Kolkata")
current_time = datetime.now(ist)

#Visible During 3:00 PM(15) to 5:00 PM(17)
if 15 <= current_time.hour < 17:
 App_df['Last Updated']=pd.to_datetime(App_df['Last Updated'],errors='coerce')
 App_df['Month']=App_df['Last Updated'].dt.month.astype(int)
#Apply filters Rating>4,Size greater than 10 and Month=jan---
 filtered = App_df[
    (App_df['Rating'] >= 4.0) &
    (App_df['Size'] >= 10) &
    (App_df['Month'] == 1)
 ].copy()
# Group by category use of agreegation
 grouped = filtered.groupby('Category').agg(
    Avg_Rating=('Rating', 'mean'),
    Total_Reviews=('Reviews', 'sum'),
    Total_Installs=('Installs', 'sum')
 ).reset_index()
#  Top 10 categories by installs 
 chart_df = grouped.sort_values('Total_Installs', ascending=False).head(10)
 chart_df['Scaled_Rating'] = chart_df['Avg_Rating'] * 100
#  Dual-axis grouped graph
 fig11 = make_subplots(specs=[[{"secondary_y": True}]])
 fig11.add_trace(
               go.Bar( 
                     x=chart_df['Category'],
                     y=chart_df['Total_Reviews'],
                     offsetgroup=1,                                        
                     marker_color='#1f77b4',
                     name="Total Reviews",
                     ),
                 secondary_y=False
             )
 fig11.add_trace(
                  go.Bar( 
                    x=chart_df['Category'],
                    y=chart_df['Avg_Rating'],
                    offsetgroup=2,              
                    marker_color='#ff7f0e',
                    name="Avarage Ratings",    
                     ),
                 secondary_y=True
               )           
 fig11.update_layout(
    title_text="<b>The Categaries wise Review and Avgrage Ratings of Appplications</b>",
    xaxis_title="Applictions Categories ",
    barmode='group',
    width=plot_width,
    height=plot_height,
    template='plotly_dark',             # Changes default fonts and grids to dark mode
    paper_bgcolor='#000000',            # Sets the outer margin background to pure black
    plot_bgcolor='#000000'
 )

 fig11.update_xaxes(title_text="Categories Of Applications")
 fig11.update_yaxes(title_text="Reviews Of Applications", secondary_y=False)
 fig11.update_yaxes(title_text="Rating Score (1-5)", secondary_y=True, range=[0, 5.5])
 #fig11.show()
 save_plot_as_html(fig11, "Stacked Group Bar of Categaries Vs Reviews and Avarage Rating.html", "The top categories on the Play Store Showing the reviews and Avgarge Ratings of Applications")
else:
 print("Graph is not Availble during this time slot.Graph will avaible between 3.PM to 5.PM  only")

Graph is not Availble during this time slot.Graph will avaible between 3.PM to 5.PM  only


In [24]:
#Task 2  Interactive Choropleth map using Plotly to visualize global installs by Category.
#For Choropleth map locations are requires but locations not specified in given dataset.
#So all the filterration of data are done according to the requirements as follows.
import pytz
from datetime import datetime
ist = pytz.timezone("Asia/Kolkata")
current_time = datetime.now(ist)
#current_time = datetime.now(ist)
# Graph Visible between 6 PM (18:00) and 8 PM (20:00)
if 18 <= current_time.hour < 20:
# Filter category not started with "A", "C", "G", "S" Using ~ operator.
  filtered_df1 = App_df[
                       ~App_df["Category"].str.startswith(("A", "C", "G", "S"))
                       ].copy()
#Filter the top 5 Category by installation
  top_5_categories=(filtered_df1.groupby("Category")["Installs"]                          
                      .sum()
                      .nlargest(5)
                      .index.tolist()
                 )
 
  filtered_df1 = filtered_df1[filtered_df1["Category"].isin(top_5_categories)]
  filtered_df1['Installs'] = filtered_df1['Installs'].apply(lambda x: '> 1M Installs' if x > 1000000 else '≤ 1M Installs')
  print(filtered_df1)
  """fig = px.choropleth(
        filtered_df,
        locations="Country",
        locationmode="country names",
        color="Highlight_Group",
        hover_data=["Category", "Installs"],
        title="Global Installs by App Category (6 PM - 8 PM IST)",
        color_discrete_map={
            "> 1M Installs": "#EF553B",
            "<= 1M Installs": "#636EFA",
        },
     )"""
  #fig.writehtml("Choropleth Map for Top Category(6 PM - 8PM IST)")   
else:
  print("Dashboard Map only visible between 6 PM and 8 PM IST.")
   


Dashboard Map only visible between 6 PM and 8 PM IST.


In [25]:
#Task 3.A Dual-axis chart comparing the average installs and revenue for free vs. paid apps within the top 3 app categories.
import re
import plotly.graph_objects as go
import pytz
from datetime import datetime
ist = pytz.timezone("Asia/Kolkata")
current_time = datetime.now(ist)

# Time filter 1 PM (13:00) and 2 PM (14:00)
if 13 <= current_time.hour < 14:

 def clean_android_version(version_str):
    if pd.isna(version_str):
        return None
    match = re.search(r"\d+(\.\d+)?", str(version_str))
    if match:
       return float(match.group()) #Returns output 4.0
    return None  # Handles text like 'Varies with device'
 #Apply the function to clean the text column into a float column
 App_df["Android Ver_num"] = App_df["Android Ver"].apply(
                                                        clean_android_version
                                                       )
# Add a column to show the revenue of application
 App_df['Revenue']=App_df['Price']*App_df['Installs']
 filtered_df2 =App_df[
        (App_df["Installs"] >= 10000)
        & (App_df["Revenue"] >= 10000)
        & (App_df["Android Ver_num"] > 4.0)
        & (App_df["Size"] > 15)
        & (App_df["Content Rating"] == "Everyone")
        & (App_df["App"].str.len() <= 30)
    ].copy()
 top_3Categories=filtered_df2['Category'].value_counts().nlargest(3).index
 chart_df=filtered_df2[filtered_df2['Category'].isin(top_3Categories)]

# Compute Mean of Installs and Revenue  Grouped by Category and Free vs Paid
 Finalchart_data=( 
           chart_df.groupby(["Category","Type"])[["Installs","Revenue"]].mean().reset_index()
           )    
#For X-Axis Values
 Finalchart_data["X_Axis"] = (Finalchart_data["Category"] + " (" + Finalchart_data["Type"] + ")")
 # Plotly Dual-Axis Interactive Chart 
 fig13= go.Figure()

    # Left Y-Axis: Average Installs (Bar Chart)
 fig13.add_trace(
         go.Bar(
                 x=Finalchart_data["X_Axis"],
                 y=Finalchart_data["Installs"],
                 name="Avg Installs",
                 marker_color="rgb(55, 135, 215)",
                 yaxis="y1",
               )
             )

    # Right Y-Axis: Average Revenue (Line Chart)
 fig13.add_trace(
        go.Scatter(
            x=Finalchart_data["X_Axis"],
            y=Finalchart_data["Revenue"],
            name="Avg Revenue ($)",
            mode="lines+markers",
            marker=dict(size=10),
            line=dict(color="rgb(255,99, 132)", width=3),
            yaxis="y2",
              )
            )

    # Layout Configuration for Dual Axis
 fig13.update_layout(
        width=plot_width,
        height=plot_height,
        plot_bgcolor="black", # Set overall chart background colors to black
        paper_bgcolor="black",
        font=dict(color="white"),
        title="Average Installs vs Revenue within Top 3 Categories for Free vs Paid Apps",
        xaxis=dict(title="Category & App Type"),
        yaxis=dict(
            title=dict(
                        text="Average Installs", font=dict(color="rgb(55, 83, 109)", size=14)
                      )
        ),
        yaxis2=dict(
                     title=dict(
                                 text="Average Revenue ($)", font=dict(color="rgb(219, 64, 82)", size=14)
                               ),                 
                     overlaying="y",
                     side="right",
                  ),
                 barmode="group",
                 legend=dict(x=1.1, y=1),
                 )

# Render Chart

 save_plot_as_html(fig13,"Dual_Axis_Graph.html","The top  3 categories on the Play Store Showing the Avg.Installs and Revenues for both type of grapn Paid And Free Applications")
else:
 print("Graph is only availbe only between 1 PM IST to 2 PM IST.")

Graph is only availbe only between 1 PM IST to 2 PM IST.


In [26]:
# Task 4:A time series line chart to show the trend of total installs over time. 
import pytz
import plotly.graph_objects as go
from datetime import datetime
ist_timezone = pytz.timezone('Asia/Kolkata')
current_time_ist = datetime.now(ist_timezone)
current_hour = current_time_ist.hour

# 18 = 6 PM, 21 = 9 PM
if not (18 <= current_hour < 21):
    print("DashBoard is visible Only Between 6PM and 9 PM")
else:    
    df=App_df.copy()
    df['Category'] = df['Category'].astype(str).str.strip()
    df['App'] = df['App'].astype(str).str.strip()
    # 1. Select Category starting with E, C, or B 
    df= df[df['Category'].str.upper().str.startswith(('E', 'C', 'B','e','c','b'))]
    # 2.App Names starting with X, Y, Z 
    df = df[~df['App'].str.upper().str.startswith(('X', 'Y', 'Z'))]
    # 3. Reviews should be  > 500 
    df['Reviews'] = pd.to_numeric(df['Reviews'], errors='coerce')
    df = df[df['Reviews'] > 500]
    # 4 App Names containing 'S'
    df= df[~df['App'].str.upper().str.contains('S')]
    # 5 Apply Category Translations of categories Beuty in HINDI,Bussiness in Tamil and Dating in German
    def translate_cat(cat):
        cat_upper = cat.upper()
        if 'BEAUTY' in cat_upper:
            return 'सुंदरता'
        elif 'BUSINESS' in cat_upper:
            return 'வணிகம்'
        elif 'DATING' in cat_upper:
            return 'das Dating'
        return cat

    df['Category'] = df['Category'].apply(translate_cat)
     
    # Ensure Date column is datetime
    df['Last Updated'] = pd.to_datetime(df['Last Updated'])

    # Aggregate total installs according to last Updated by Category
    df_chart= df.groupby(['Last Updated', 'Category'])['Installs'].sum().reset_index()
    print(df_chart['Category'].unique())
    # Build Plotly Graph ---
    fig14 = go.Figure()
    shaded_periods = set()
    # ALL unique categories to showi on  plot.
    for category in df_chart['Category'].unique():
        cat_df = df_chart[df_chart['Category'] == category].sort_values('Last Updated').copy()
              
        # Calculate Month-over-Month Growth Rate
        cat_df['mom_growth'] = cat_df['Installs'].pct_change()
        # Add trace inside the category loop
        fig14.add_trace(go.Scatter(
            x=cat_df['Last Updated'], 
            y=df_chart['Installs'],
            mode='lines+markers',
            textfont_size=10,
            text=df_chart['Installs'].apply(lambda x: f"{x:,}"),
            textposition="top center",
            name=category,
            line=dict(width=3),
            marker=dict(size=8)
        ))

        # MoM growth > 20% coordinates
        for i in range(1, len(cat_df)):
            if cat_df['mom_growth'].iloc[i] > 0.20:
                start_date = cat_df['Last Updated'].iloc[i-1]
                end_date = cat_df['Last Updated'].iloc[i]
                shaded_periods.add((start_date, end_date))

    # Shaded rectangles
    shapes = []
    for start, end in shaded_periods:
        shapes.append(dict(
            type="rect",
            xref="x",
            yref="paper",
            x0=start,
            y0=0,
            x1=end,
            y1=1,
            fillcolor="rgba(34, 197, 94, 0.18)",
            layer="below",
            line=dict(width=0)
            
        ))

    fig14.update_layout(
        width=plot_width,
        height=plot_height,
        font=dict(color="white"),
        title="<b>Total App Installs Vs Time</b><br><sup>Shaded regions shows MoM growth 20%</b></sup>",
        xaxis_title="Timeline",
        yaxis_title="Total Installs",
        shapes=shapes,
        plot_bgcolor="#E5E5E5",   # Gray background inside the chart area
        paper_bgcolor="black",
        template="plotly_white",
        hovermode="x unified"    
    )

    
    click_js_script = """
           var graphDiv = document.getElementById('{plot_id}');
           graphDiv.on('plotly_click', function(data){
          // Optional: Get details of the clicked point if needed
         var point = data.points[0];
         var clicked_date = point.x;
         var category_name = point.data.name;
    
        // Open the separate HTML file in a new tab
        window.open('other_graph.html', '_blank');
       });
     """

# 2. Save the file with the included JavaScript post-script
    fig14.write_html(
         "Time_series_chart.html", 
          include_plotlyjs="inline", 
           post_script=click_js_script
          )
    #fig14.show() 
    save_plot_as_html(fig14,"Time_series_chart.html","A time series line chart tshowing the trend of total installs over time, segmented by app category and Highlight periods of significant growth by shading the areas under the curve where the increase in installs exceeds 20% month-over-month.")

DashBoard is visible Only Between 6PM and 9 PM


In [27]:
#Task 5:Abubble  chart to analyze the relationship between app size (in MB) and average rating.
from datetime import datetime
import zoneinfo
ist_zone = zoneinfo.ZoneInfo("Asia/Kolkata")
now_ist = datetime.now(ist_zone)
current_hour = now_ist.hour
# Graph visble (5:00 PM to 7:00 PM)
if ( 17 <= current_hour < 19):
    # 2.Create a Data Frame
    df1 = App_df.copy()
    selcategories=["Game", "Beauty","Business", "commics", "commication","Dating","Entertainment","social","event"]
    selcategories_upper=[cat.upper() for cat in selcategories]
    # 3. Apply criteria filters
    # - Rating > 3.5 ,Reviews > 500 , Installs > 50,000, Sentiment Subjectivity > 0.5
    # - App name must NOT contain the letter "S" or "s"
    filtered_df1 = df1[
        (df1["Category"].str.upper().isin(selcategories_upper))
        & (df1["Rating"] > 3.5)
        & (df1["Reviews"] > 500)
        & (df1["Installs"] > 50000)
        & (df_review["Sentiment_Subjectivity"] > 0.5)
        & (~df1["App"].str.contains("S", case=False, na=False))
    ].copy()
    
    # 4. Map Translations for requested categories
    translation_map = {
        "BEAUTY": "सौंदर्य",  # Hindi
        "BUSINESS": "வணிகம்",  # Tamil
        "DATING": "Dating",  # German
    }
    filtered_df1["Category"] = filtered_df1["Category"].replace(translation_map)
    # Update categories list to match updated names for mapping color scales
    updated_categories = [translation_map.get(c, c) for c in selcategories]
    # Game category mapped to Pink
    base_colors = px.colors.qualitative.Safe
    color_discrete_map = {}
    color_index = 0

    for cat in updated_categories:
        if cat == "Game":
            color_discrete_map[cat] = "#FF69B4"  # Hot Pink
        else:
            color_discrete_map[cat] = base_colors[color_index % len(base_colors)]
            color_index += 1

    # 6. Render Bubble Chart
    if filtered_df1.empty:
        print(
            "No applications currently match all criteria configurations simultaneously."
        )
    else:
        fig15 = px.scatter(
            filtered_df1,
            x="Size",
            y="Rating",
            size="Installs",
            color="Category",
            hover_name="App",
            size_max=50,
            color_discrete_map=color_discrete_map,
            category_orders={"Category": updated_categories},
            title="App Performance Index (Size vs Rating)",
            labels={
                "Size_MB": "App Size (MB)",
                "Rating": "Average Rating",
                "Category": "App Category",
            },
        )

        fig15.update_layout(
            width=plot_width,
            height=plot_height,
            font=dict(color="white"),
            plot_bgcolor="#E5E5E5",
            paper_bgcolor="black",
            template="plotly_white",
            title_font_size=20,
            xaxis_title_font_size=14,
            yaxis_title_font_size=14,
        )

       # fig15.show()
        save_plot_as_html(fig15," Bubble_Chart_AvgVsAppSize.html","BUbble chart showing app size vs average rating, with the bubble size showing the number of installs for Category")
else:
    print(
        f"Dashboard Status:The requested chart is only visible between 5:00 PM and 7:00 PM IST.Current time: {now_ist.strftime('%I:%M %p %Z')}."
    )

Dashboard Status:The requested chart is only visible between 5:00 PM and 7:00 PM IST.Current time: 12:19 PM IST.


In [28]:
#Task6:A stacked area chart to visualize the cumulative number of installs over time.
from datetime import datetime
import zoneinfo
import plotly.graph_objects as go


ist_zone = zoneinfo.ZoneInfo("Asia/Kolkata")
now_ist = datetime.now(ist_zone)
current_hour = now_ist.hour
df3=App_df.copy()
# 1.Graph visible from 4:00 PM to 6:00 PM)
if ( 16 <= current_hour < 18):
# Filtering the data according to the requirement 
 df3 = df3[(df3["Rating"] >= 4.2)
        & (~df3["App"].str.contains(r"\d", na=False))  # No numbers in name
        & (df3["Category"].str.upper().str.startswith(("T", "P")))  # Starts with T or P
        & (df3["Reviews"] > 1000)
        & (df3["Size"].between(20, 80))  # Size between 20 MB and 80 MB
        ].copy()

# Data Aggrigation
 df3['Last Updated'] = pd.to_datetime(df3['Last Updated'])
 monthly_df = df3.groupby([df3['Last Updated'].dt.to_period('M'), 'Category'])['Installs'].sum().reset_index()
 monthly_df['Last Updated'] = monthly_df['Last Updated'].dt.to_timestamp()
# Pivot table for  calculate Monthly installation
 pivot_df = monthly_df.pivot(index='Last Updated', columns='Category', values='Installs').fillna(0)
# Month-over-Month growth percentage matrix
 mom_growth = pivot_df.pct_change().fillna(0)
# Cumulative Installs matrix
 cumulative_df = pivot_df.cumsum()
# Translations mapping according to the Requirements
 legend_translations = {
    "TRAVEL_AND_LOCAL":"Voyages et local", # French
    "PRODUCTIVITY": "Productividad",     # Spanish
    "PHOTOGRAPHY": "写真",               # Japanese
    "PARENTING": "Éducation des enfants"  # Handled safely
}

# Color styling library mapped cleanly to uppercase
 color_palette = {
    "TRAVEL_AND_LOCAL": {"normal": "rgba(20, 90, 50, 0.55)",  "high": "rgba(11, 57, 31, 0.98)"},   # Dark Forest Green
    "PRODUCTIVITY":   {"normal": "rgba(21, 67, 96, 0.55)",  "high": "rgba(11, 38, 57, 0.98)"},   # Dark Navy Blue
    "PHOTOGRAPHY":    {"normal": "rgba(160, 64, 0, 0.55)",  "high": "rgba(100, 30, 0, 0.98)"},   # Burnt Orange
    "PARENTING":      {"normal": "rgba(81, 46, 95, 0.55)",  "high": "rgba(48, 25, 58, 0.98)"}    # Dark Plum/Purple
  }

 fig16 = go.Figure()
 categories = cumulative_df.columns
 dates = cumulative_df.index

# Build custom area chart 
 for cat in categories:
    # Use exact string or capital fallback
    display_name = legend_translations.get(cat.upper(), cat.title())
    
    # Define stacked area 
    current_index = list(categories).index(cat)
    if current_index == 0:
        baseline = np.zeros(len(dates))
    else:
        baseline = cumulative_df.iloc[:, :current_index].sum(axis=1).values
        
    y_values = baseline + cumulative_df[cat].values
    
    # Plot trace month by month to apply color intensity logic
    for i in range(len(dates) - 1):
        # Calculate localized monthly growth rate
        growth_rate = mom_growth[cat].iloc[i+1]
        
        # Pull color map configurations using unified uppercase string matching
        category_colors = color_palette.get(
            cat.upper(), 
            {"normal": "rgba(128, 128, 128, 0.35)", "high": "rgba(128, 128, 128, 0.95)"} # Grey safety backup
        )
        
        # Highlight segment if MoM growth > 25%
        fill_color = category_colors["high"] if growth_rate > 0.25 else category_colors["normal"]
      
        fig16.add_trace(go.Scatter(
            x=dates[i:i+2],
            y=y_values[i:i+2],
            mode='lines',
            line=dict(width=0.5, color='rgba(0,0,0,0)'),
            fill='tonexty' if current_index > 0 else 'tozeroy',
            fillcolor=fill_color,
            name=display_name,
            legendgroup=cat,
            showlegend=True if i == 0 else False, # Prevents repetitive legend duplicate entries
            hovertemplate=f"<b>{display_name}</b><br>Date: %{{x|%B %Y}}<br>Cumulative Installs: %{{y:,.0f}}<br>MoM Growth: {growth_rate:.1%}<extra></extra>"
        ))

# Final Graph
 fig16.update_layout(
    title="<b> Cummulative App Installs vs Date</b><br> Dataset Analysis",
    xaxis_title="Timeline",
    yaxis_title="Cumulative Installs per DATE",
    hovermode="x unified",
    legend_title_text="App Category",
    font=dict(color="black"),
    paper_bgcolor="#EAEAEA",  # Slate dark gray
    plot_bgcolor="white",   # Keeps the background seamless,
    template="plotly_white",
    margin=dict(l=40, r=40, t=60, b=40),
    width=plot_width,
    height=plot_height
 )

#fig16.show()
 save_plot_as_html(fig16,"Cummulative_App_Vs_Time.html","A stacked area chart to visualize the cumulative number of installs over time for each app category")
else:
 print("Graph is only availbe only between 4 PM IST to 6 PM IST.")

Graph is only availbe only between 4 PM IST to 6 PM IST.


In [29]:
# Category Analysis Plot
category_counts = App_df['Category'].value_counts().nlargest(10)
fig1 = px.bar(
    x=category_counts.index,
    y=category_counts.values,
    labels={'x': 'Category', 'y': 'Count'},
    title='Top Categories on Play Store',
    color=category_counts.index,
    color_discrete_sequence=px.colors.sequential.Plasma,
    width=plot_width,
    height=plot_height
)
fig1.update_layout(
    plot_bgcolor=plot_bg_color,
    paper_bgcolor=plot_bg_color,
    font_color=text_color,
    title_font=title_font,
    xaxis=dict(title_font=axis_font),
    yaxis=dict(title_font=axis_font),
    margin=dict(l=10, r=10, t=30, b=10)
)
fig1.update_traces(marker=dict(line=dict(color=text_color, width=1)))
save_plot_as_html(fig1, "Category_Analysis.html", "The top categories on the Play Store are dominated by tools, entertainment, and productivity apps. This suggests users are looking for apps that either provide utility or offer leisure activities.")


In [30]:
 #Type Analysis Plot
type_counts = App_df['Type'].value_counts()
fig2 = px.pie(
    values=type_counts.values,
    names=type_counts.index,
    title='App Type Distribution',
    color_discrete_sequence=px.colors.sequential.RdBu,
    width=plot_width,
    height=plot_height
)
fig2.update_traces(textposition='inside', textinfo='percent+label')
fig2.update_layout(
    plot_bgcolor=plot_bg_color,
    paper_bgcolor=plot_bg_color,
    font_color=text_color,
    title_font=title_font,
    margin=dict(l=10, r=10, t=30, b=10)
)
save_plot_as_html(fig2, "App_Type_analysis.html", "Most apps on the Play Store are free, indicating a strategy to attract users first and monetize through ads or in-app purchases.")

In [31]:
#Rating Distribution Plot
fig3 = px.histogram(
    App_df,
    x='Rating',
    nbins=20,
    title='Rating Distribution',
    color_discrete_sequence=['#636EFA'],
    width=plot_width,
    height=plot_height
)
fig3.update_layout(
    plot_bgcolor=plot_bg_color,
    paper_bgcolor=plot_bg_color,
    font_color=text_color,
    title_font=title_font,
    xaxis=dict(title_font=axis_font),
    yaxis=dict(title_font=axis_font),
    margin=dict(l=10, r=10, t=30, b=10)
)
save_plot_as_html(fig3, "App_Rating_distribution.html", "Ratings are skewed towards higher values, suggesting that most apps are rated favorably by users.")

In [32]:
sentiment_counts = df_review['Sentiment_Score'].value_counts()
fig4 = px.bar(
    x=sentiment_counts.index,
    y=sentiment_counts.values,
    labels={'x': 'Sentiment Score', 'y': 'Count'},
    title='Sentiment Distribution',
    color=sentiment_counts.index,
    color_discrete_sequence=px.colors.sequential.RdPu,
    width=plot_width,
    height=plot_height
)
fig4.update_layout(
    plot_bgcolor=plot_bg_color,
    paper_bgcolor=plot_bg_color,
    font_color=text_color,
    title_font=title_font,
    xaxis=dict(title_font=axis_font),
    yaxis=dict(title_font=axis_font),
    margin=dict(l=10, r=10, t=30, b=10)
)
fig4.update_traces(marker=dict(line=dict(color=text_color, width=1)))
save_plot_as_html(fig4, "App_Sentiment_distribution.html", "Sentiments in reviews show a mix of positive and negative feedback.")

In [33]:
#Installs by Category Plot
installs_by_category = App_df.groupby('Category')['Installs'].sum().nlargest(10)
print()
fig5 = px.bar(
    x=installs_by_category.values,
    y=installs_by_category.index,
    orientation='h',
    labels={'x': 'Installs', 'y': 'Category'},
    title='Installs by Category',
    color=installs_by_category.index,
    color_discrete_sequence=px.colors.sequential.Blues,
    width=plot_width,
    height=plot_height
)
fig5.update_layout(
    plot_bgcolor=plot_bg_color,
    paper_bgcolor=plot_bg_color,
    font_color=text_color,
    title_font=title_font,
    xaxis=dict(title_font=axis_font),
    yaxis=dict(title_font=axis_font),
    margin=dict(l=10, r=10, t=30, b=10)
)
fig5.update_traces(marker=dict(line=dict(color=text_color, width=1)))
save_plot_as_html(fig5, "App_Installs_by_category.html", "The categories with the most installs are social and communication apps, which reflects their broad appeal and daily usage.")

In [34]:
# Updates Per Year Plot
updates_per_year = App_df['Last Updated'].dt.year.value_counts().sort_index()
fig6 = px.line(
    x=updates_per_year.index,
    y=updates_per_year.values,
    labels={'x': 'Year', 'y': 'Number of Updates'},
    title='Number of Updates Over the Years',
    color_discrete_sequence=['#AB63FA'],
    width=plot_width,
    height=plot_height
)
fig6.update_layout(
    plot_bgcolor=plot_bg_color,
    paper_bgcolor=plot_bg_color,
    font_color=text_color,
    title_font=title_font,
    xaxis=dict(title_font=axis_font),
    yaxis=dict(title_font=axis_font),
    margin=dict(l=10, r=10, t=30, b=10)
)
save_plot_as_html(fig6, "Number_of_Updatesby_year.html", "Updates have been increasing over the years.")

In [35]:
#Figure 7
installs_by_category=App_df.groupby('Category')['Revenue'].sum().nlargest(10)
fig7=px.bar(
    x=installs_by_category.index,
    y=installs_by_category.values,
    labels={'x':'Category','y':'Revenue'},
    title='Revenue by Category',
    color=installs_by_category.index,
    color_discrete_sequence=px.colors.sequential.Greens,
    width=plot_width,
    height=plot_height
)
fig7.update_layout(
    plot_bgcolor='black',
    paper_bgcolor='black',
    font_color='white',
    title_font={'size':16},
    xaxis=dict(title_font={'size':12}),
    yaxis=dict(title_font={'size':12}),
    margin=dict(l=10,r=10,t=30,b=10)
)

save_plot_as_html(fig7,"Revenue Graph 7.html","Categories such as Business and Productivity lead in revenue generation, indicating their monetization potential")

In [36]:
#Figure 8
genre_counts=App_df['Genres'].str.split(';',expand=True).stack().value_counts().nlargest(10)
fig8=px.bar(
    x=genre_counts.index,
    y=genre_counts.values,
    labels={'x':'Genre','y':'Count'},
    title='Top Genres',
    color=installs_by_category.index,
    color_discrete_sequence=px.colors.sequential.OrRd,
    width=plot_width,
    height=plot_height
)
fig8.update_layout(
    plot_bgcolor='black',
    paper_bgcolor='black',
    font_color='white',
    title_font={'size':16},
    xaxis=dict(title_font={'size':12}),
    yaxis=dict(title_font={'size':12}),
    margin=dict(l=10,r=10,t=30,b=10)
)
save_plot_as_html(fig8,"Genre Graph 8.html","Action and Casual genres are the most common, reflecting users.")

In [37]:
#Figure 9
fig9=px.scatter(
    App_df,
    x='Last Updated',
    y='Rating',
    color='Type',
    title='Impact of Last Update on Rating',
    color_discrete_sequence=px.colors.qualitative.Vivid,
    width=plot_width,
    height=plot_height
)
fig9.update_layout(
    plot_bgcolor='black',
    paper_bgcolor='black',
    font_color='white',
    title_font={'size':16},
    xaxis=dict(title_font={'size':12}),
    yaxis=dict(title_font={'size':12}),
    margin=dict(l=10,r=10,t=30,b=10)
)
save_plot_as_html(fig9,"Update Graph 9.html","The Scatter Plot shows a weak correlation between the last update and ratings, suggesting that more frequent updates dont always result in better ratings.")

In [38]:
#Figure 10
fig10=px.box(
    App_df,
    x='Type',
    y='Rating',
    color='Type',
    title='Rating for Paid vs Free Apps',
    color_discrete_sequence=px.colors.qualitative.Pastel,
    width=plot_width,
    height=plot_height
)
fig10.update_layout(
    plot_bgcolor='black',
    paper_bgcolor='black',
    font_color='white',
    title_font={'size':16},
    xaxis=dict(title_font={'size':12}),
    yaxis=dict(title_font={'size':12}),
    margin=dict(l=10,r=10,t=30,b=10)
)

save_plot_as_html(fig10,"Paid Free Graph 10.html","Paid apps generally have higher ratings compared to free apps, suggesting that users expect higher quality from apps they pay for")

In [39]:
plot_containers_split=plot_containers.split('</div>')

In [40]:
if len(plot_containers_split) > 1:
    final_plot=plot_containers_split[-2]+'</div>'
else:
    final_plot=plot_containers

In [41]:
dashboard_html= """
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name=viewport" content="width=device-width,initial-scale-1.0">
    <title> Google Play Store Review Analytics</title>
    <style>
        body {{
            font-family: Arial, sans-serif;
            background-color: #333;
            color: #fff;
            margin: 0;
            padding: 0;
        }}
        .header {{
            display: flex;
            align-items: center;
            justify-content: center;
            padding: 20px;
            background-color: #444
        }}
        .header img {{
            margin: 0 10px;
            height: 50px;
        }}
        .container {{
            display: flex;
            flex-wrap: wrap;
            justify_content: center;
            padding: 20px;
        }}
        .plot-container {{
            border: 2px solid #555
            margin: 10px;
            padding: 10px;
            width: {plot_width}px;
            height: {plot_height}px;
            overflow: hidden;
            position: relative;
            cursor: pointer;
        }}
        .insights {{
            display: none;
            position: absolute;
            right: 10px;
            top: 10px;
            background-color: rgba(0,0,0,0.7);
            padding: 5px;
            border-radius: 5px;
            color: #fff;
        }}
        .plot-container: hover .insights {{
            display: block;
        }}
        </style>
        <script>
            function openPlot(filename) {{
                window.open(filename, '_blank');
                }}
        </script>
    </head>
    <body>
        <div class= "header">
            <img src="https://upload.wikimedia.org/wikipedia/commons/thumb/4/4a/Logo_2013_Google.png/800px-Logo_2013_Google.png" alt="Google Logo">
            <h1>Google Play Store Reviews Analytics</h1>
            <img src="https://upload.wikimedia.org/wikipedia/commons/thumb/7/78/Google_Play_Store_badge_EN.svg/1024px-Google_Play_Store_badge_EN.svg.png" alt="Google Play Store Logo">
        </div>
        <div class="container">
            {plots}
        </div>
    </body>
    </html>
    """


In [42]:

final_html=dashboard_html.format(plots=plot_containers,plot_width=plot_width,plot_height=plot_height)

In [43]:
dashboard_path=os.path.join(html_files_path,"web ospage.html")

In [44]:
with open(dashboard_path, "w", encoding="utf-8") as f:
    f.write(final_html)

In [45]:
webbrowser.open('file://'+os.path.realpath(dashboard_path))

True